## Label the dataset using our trained model

In [1]:
from ultralytics import YOLO
import torch
from torchvision import models, transforms
from PIL import Image
import os

# Load YOLO detector
yolo_model = YOLO(r"D:\IITBHU Internship\code\runs\detect\train11\best.pt")

# Load CNN classifier
cnn_model = models.resnet18()
cnn_model.fc = torch.nn.Linear(cnn_model.fc.in_features, 2)  # drone / non_drone
cnn_model.load_state_dict(torch.load("drone_classifier.pth", map_location="cpu"))
cnn_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn_model.to(device)

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

classes = ["drone","non_drone"]

# Folder with unlabeled images
img_dir = r"D:\IITBHU Internship\code\DroneDatasetCombined\outliers\images"
save_dir = r"D:\IITBHU Internship\code\DroneDatasetCombined\outliers\labels"
os.makedirs(save_dir, exist_ok=True)

for fname in os.listdir(img_dir):
    if not fname.lower().endswith((".jpg",".jpeg",".png")):
        continue

    img_path = os.path.join(img_dir, fname)
    results = yolo_model(img_path, conf=0.25, device=0, verbose=False)[0]

    label_lines = []
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        # Crop region for CNN
        crop = Image.open(img_path).convert("RGB").crop((x1,y1,x2,y2))
        crop_tensor = transform(crop).unsqueeze(0).to(device)

        with torch.no_grad():
            output = cnn_model(crop_tensor)
            _, pred = torch.max(output, 1)
            label = classes[pred.item()]

        # Only save YOLO label if CNN confirms "drone"
        if label == "drone":
            # Convert back to YOLO normalized format
            w, h = results.orig_shape[1], results.orig_shape[0]
            xc = ((x1+x2)/2)/w
            yc = ((y1+y2)/2)/h
            bw = (x2-x1)/w
            bh = (y2-y1)/h
            label_lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    # Always create a .txt file (empty if no drone detected)
    label_path = os.path.join(save_dir, os.path.splitext(fname)[0] + ".txt")
    with open(label_path, "w") as f:
        if label_lines:
            f.write("\n".join(label_lines))
        # else: leave file empty → means no drone in this image

    print(f"Processed {fname}, labels: {len(label_lines)}")

C:\Users\100ra\AppData\Local\Temp\ipykernel_35056\2124352271.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load("drone_classifier.pth"

Processed drone_img_001.JPEG, labels: 1
Processed drone_img_002.JPEG, labels: 1
Processed drone_img_003.JPEG, labels: 2
Processed drone_img_004.JPEG, labels: 1
Processed drone_img_005.JPEG, labels: 1
Processed drone_img_006.JPEG, labels: 1
Processed drone_img_007.JPEG, labels: 1
Processed drone_img_008.JPEG, labels: 3
Processed drone_img_009.JPEG, labels: 2
Processed drone_img_010.JPEG, labels: 0
Processed drone_img_011.JPEG, labels: 1
Processed drone_img_012.JPEG, labels: 1
Processed drone_img_013.JPEG, labels: 1
Processed drone_img_014.JPEG, labels: 2
Processed drone_img_015.JPEG, labels: 1
Processed drone_img_016.JPEG, labels: 1
Processed drone_img_017.JPEG, labels: 1
Processed drone_img_018.JPEG, labels: 1
Processed drone_img_019.JPEG, labels: 1
Processed drone_img_020.JPEG, labels: 0
Processed drone_img_021.JPEG, labels: 1
Processed drone_img_022.JPEG, labels: 2
Processed drone_img_023.JPEG, labels: 1
Processed drone_img_024.JPEG, labels: 1
Processed drone_img_025.JPEG, labels: 1


## **Code for rename the image and label file name**

In [1]:
import os

def rename_yolo_dataset(img_dir, label_dir, prefix="drone_img_", start_num=1):
    """
    Rename images and labels in YOLO dataset format.
    
    Args:
        img_dir (str): Path to images folder.
        label_dir (str): Path to labels folder.
        prefix (str): Prefix for new filenames.
        start_num (int): Starting number for renaming.
    """
    # Get sorted list of image files
    img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif'))])
    
    counter = start_num
    for img_file in img_files:
        # Old paths
        old_img_path = os.path.join(img_dir, img_file)
        old_label_path = os.path.join(label_dir, os.path.splitext(img_file)[0] + ".txt")
        
        # New names
        new_name = f"{prefix}{counter:03d}"   # zero-padded e.g. drone_img_001
        new_img_path = os.path.join(img_dir, new_name + os.path.splitext(img_file)[1])
        new_label_path = os.path.join(label_dir, new_name + ".txt")
        
        # Rename image
        os.rename(old_img_path, new_img_path)
        
        # Rename label if exists
        if os.path.exists(old_label_path):
            os.rename(old_label_path, new_label_path)
        
        print(f"Renamed: {img_file} -> {new_name}")
        counter += 1


In [9]:
img_dirt = r"D:\IITBHU Internship\other datasets\BirdVsDrone2\drone"
label_dirt = r"D:\IITBHU Internship\other datasets\BirdVsDrone2\Drones_labels"


# Example usage:
rename_yolo_dataset(img_dirt, label_dirt, prefix="drone_img_", start_num=1572)

Renamed: 000000000000.jpg -> drone_img_1572
Renamed: 000000000001.jpg -> drone_img_1573
Renamed: 000000000002 - Copy - Copy.jpg -> drone_img_1574
Renamed: 000000000002 - Copy.jpg -> drone_img_1575
Renamed: 000000000002.jpg -> drone_img_1576
Renamed: 000000000003.jpg -> drone_img_1577
Renamed: 000000000004 - Copy - Copy.jpg -> drone_img_1578
Renamed: 000000000004 - Copy.jpg -> drone_img_1579
Renamed: 000000000004.jpg -> drone_img_1580
Renamed: 000000000005 - Copy - Copy.jpg -> drone_img_1581
Renamed: 000000000005 - Copy.jpg -> drone_img_1582
Renamed: 000000000005.jpg -> drone_img_1583
Renamed: 000000000007.jpg -> drone_img_1584
Renamed: 000000000009.jpg -> drone_img_1585
Renamed: 000000000010 - Copy - Copy.jpg -> drone_img_1586
Renamed: 000000000010 - Copy.jpg -> drone_img_1587
Renamed: 000000000010.jpg -> drone_img_1588
Renamed: 000000000011.jpg -> drone_img_1589
Renamed: 000000000012.jpg -> drone_img_1590
Renamed: 000000000013.jpg -> drone_img_1591
Renamed: 000000000016.jpg -> drone_i

## checking the lebel

In [1]:
import cv2
import os

# Paths
img_dir =  r"D:\IITBHU Internship\code\DroneDatasetCombined\outliers\images"
label_dir =  r"D:\IITBHU Internship\code\DroneDatasetCombined\outliers\labels"

# Image size (YOLO labels are normalized)
IMG_EXTS = [".jpg", ".jpeg", ".png"]

# Class names (adjust to your dataset)
class_names = ["drone"]

for fname in os.listdir(img_dir):
    if not any(fname.lower().endswith(ext) for ext in IMG_EXTS):
        continue

    img_path = os.path.join(img_dir, fname)
    label_path = os.path.join(label_dir, os.path.splitext(fname)[0] + ".txt")

    # Load image
    img = cv2.imread(img_path)
    if img is None:
        print("❌ Could not read:", img_path)
        continue
    h, w = img.shape[:2]

    # Draw labels if file exists
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id, x_center, y_center, bw, bh = map(float, parts)

                # Convert YOLO normalized coords → pixel coords
                x_center, y_center, bw, bh = x_center * w, y_center * h, bw * w, bh * h
                x1 = int(x_center - bw/2)
                y1 = int(y_center - bh/2)
                x2 = int(x_center + bw/2)
                y2 = int(y_center + bh/2)

                # Draw bounding box
                cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
                label = class_names[int(cls_id)]
                cv2.putText(img, label, (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

    # Show image
    cv2.imshow("Labeled Image", img)
    key = cv2.waitKey(0)
    if key == ord('q'):  # press 'q' to quit early
        break

cv2.destroyAllWindows()

## **Manual checking labels are correct and save else ignore**

In [13]:
import cv2
import os
import shutil

# Paths
img_dir   = r"D:\IITBHU Internship\code\DroneDatasetCombined\outliers\images"
label_dir = r"D:\IITBHU Internship\code\DroneDatasetCombined\outliers\labels"

# Output clean dataset folders
clean_img_dir   = r"D:\IITBHU Internship\code\DroneDatasetCombined\clean\images"
clean_label_dir = r"D:\IITBHU Internship\code\DroneDatasetCombined\clean\labels"
os.makedirs(clean_img_dir, exist_ok=True)
os.makedirs(clean_label_dir, exist_ok=True)

IMG_EXTS = [".jpg", ".jpeg", ".png"]
class_names = ["drone"]   # adjust if you have more classes

for fname in sorted(os.listdir(img_dir)):
    if not any(fname.lower().endswith(ext) for ext in IMG_EXTS):
        continue

    img_path   = os.path.join(img_dir, fname)
    label_path = os.path.join(label_dir, os.path.splitext(fname)[0] + ".txt")

    img = cv2.imread(img_path)
    if img is None:
        print("❌ Could not read:", img_path)
        continue
    h, w = img.shape[:2]

    # Draw labels if file exists
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id, x_center, y_center, bw, bh = map(float, parts)

                # Convert YOLO normalized coords → pixel coords
                x_center, y_center, bw, bh = x_center * w, y_center * h, bw * w, bh * h
                x1 = int(x_center - bw/2)
                y1 = int(y_center - bh/2)
                x2 = int(x_center + bw/2)
                y2 = int(y_center + bh/2)

                # Draw bounding box
                cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
                label = class_names[int(cls_id)] if int(cls_id) < len(class_names) else str(cls_id)
                cv2.putText(img, label, (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

    # Show image
    cv2.imshow("Label Checker", img)
    print(f"Image: {fname} → Press [Enter] to save, [Space] to skip, [q] to quit")

    key = cv2.waitKey(0)

    if key == ord('q'):   # quit
        break
    elif key == 13:       # Enter key → save
        shutil.copy(img_path, clean_img_dir)
        if os.path.exists(label_path):
            shutil.copy(label_path, clean_label_dir)
        print(f"✅ Saved {fname} and its label")
    elif key == 32:       # Space key → skip
        print(f"⏩ Skipped {fname}")

cv2.destroyAllWindows()

Image: drone_img_001.JPEG → Press [Enter] to save, [Space] to skip, [q] to quit
✅ Saved drone_img_001.JPEG and its label
Image: drone_img_002.JPEG → Press [Enter] to save, [Space] to skip, [q] to quit
✅ Saved drone_img_002.JPEG and its label
Image: drone_img_003.JPEG → Press [Enter] to save, [Space] to skip, [q] to quit
✅ Saved drone_img_003.JPEG and its label
Image: drone_img_004.JPEG → Press [Enter] to save, [Space] to skip, [q] to quit
✅ Saved drone_img_004.JPEG and its label
Image: drone_img_005.JPEG → Press [Enter] to save, [Space] to skip, [q] to quit
